In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run util path

In [0]:
dbutils.widgets.text("catalog", "ecommerce", "Catalog")
dbutils.widgets.text("data_source", "payments", "Data Source")

In [0]:
catalog= dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path=f'dataset path/{data_source}'
print(base_path)

In [0]:
df_bronze= spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
display(df_bronze.limit(10))

In [0]:
df_duple=df_bronze.groupBy('order_id', 'payment_type').count().filter('count > 1').display()

In [0]:
df_silver=df_bronze
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
df_silver= spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")
df_gold= df_silver.groupBy("order_id").agg(
    F.sum("payment_value").alias("total_payment_value"),
    F.max("payment_installments").alias("max_installments")
) 

In [0]:
df_gold.display()

In [0]:
df_gold=df_gold.withColumn(
    "total_payment_value",
    F.round(F.col("total_payment_value"), 2)
)

In [0]:
df_gold=df_gold.withColumnRenamed('total_payment_value', 'payment_amount')
df_gold=df_gold.withColumnRenamed('N.o of installments', 'installments')
df_gold.display()


In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")